In [ ]:
import numpy as np
import pandas as pd

# 12,000 rows ka synthetic raw dataset create karna
np.random.seed(42)
n_rows = 12000

data = {
    "Transaction_ID": [f"TXN_{i}" for i in range(1, n_rows + 1)],
    "Date": pd.date_range("2024-01-01", periods=n_rows, freq="30min")
    .astype(str)
    .values,
    "Customer_ID": np.random.choice(
        [f"CUST_{i}" for i in range(100, 500)], size=n_rows
    ),
    "Category": np.random.choice(
        ["Electronics", "Apparel", "Home Goods", None],
        size=n_rows,
        p=[0.4, 0.35, 0.2, 0.05],
    ),
    "Quantity": np.random.choice([1, 2, 3, 4, 5, -2, 999], size=n_rows),
    "Unit_Price": np.random.uniform(15.0, 600.0, size=n_rows).round(2),
    "Cost_Price": np.random.uniform(10.0, 350.0, size=n_rows).round(2),
}

df = pd.DataFrame(data)

# Real-world mess inject karna: Duplicates, null values, invalid dates
df = pd.concat([df, df.iloc[:150]], ignore_index=True)  # 150 duplicate rows
df.loc[15:45, "Unit_Price"] = np.nan  # Missing prices
df.loc[60:75, "Date"] = "invalid_timestamp"  # Corrupt date strings

# Optional: Raw copy export karna (agar record rakhna ho)
df.to_csv("raw_dataset.csv", index=False)
print("Raw dataset ready! Shape:", df.shape)

Raw dataset ready! Shape: (12150, 7)


In [ ]:
print("=== BEFORE CLEANING AUDIT ===")
print("Initial Shape:", df.shape)
print("\n--- Column Info & Data Types ---")
df.info()

print("\n--- Missing Values Count ---")
print(df.isnull().sum())

print("\n--- Duplicate Rows Count ---")
print("Duplicates:", df.duplicated().sum())

print("\n--- Outlier Check (Quantity) ---")
print(df["Quantity"].describe())

=== BEFORE CLEANING AUDIT ===
Initial Shape: (12150, 7)

--- Column Info & Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12150 entries, 0 to 12149
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Transaction_ID  12150 non-null  object 
 1   Date            12150 non-null  object 
 2   Customer_ID     12150 non-null  object 
 3   Category        11547 non-null  object 
 4   Quantity        12150 non-null  int64  
 5   Unit_Price      12119 non-null  float64
 6   Cost_Price      12150 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 664.6+ KB

--- Missing Values Count ---
Transaction_ID      0
Date                0
Customer_ID         0
Category          603
Quantity            0
Unit_Price         31
Cost_Price          0
dtype: int64

--- Duplicate Rows Count ---
Duplicates: 103

--- Outlier Check (Quantity) ---
count    12150.000000
mean       148.426996
std        3

In [ ]:
# 1. Remove duplicate records
df = df.drop_duplicates().reset_index(drop=True)

# 2. Fix corrupt dates & parse datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).reset_index(drop=True)

# 3. Handle missing values
df["Category"] = df["Category"].fillna("Unknown")

# Unit_Price missing values ko median se fill karna (per Category)
df["Unit_Price"] = df.groupby("Category")["Unit_Price"].transform(
    lambda x: x.fillna(x.median())
)

# 4. Outlier removal: Quantity valid range (1 se 20 units)
df = df[(df["Quantity"] > 0) & (df["Quantity"] <= 20)].reset_index(drop=True)

print("Cleaning complete! Rows remaining:", len(df))

Cleaning complete! Rows remaining: 8607


In [ ]:
# 1. Date features extract karna
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Month_Name"] = df["Date"].dt.strftime("%B")
df["Day_Name"] = df["Date"].dt.strftime("%A")

# 2. Financial metrics calculate karna
df["Revenue"] = (df["Quantity"] * df["Unit_Price"]).round(2)
df["Total_Cost"] = (df["Quantity"] * df["Cost_Price"]).round(2)
df["Profit"] = (df["Revenue"] - df["Total_Cost"]).round(2)

# Profit margin percentage: (Profit / Revenue) * 100
df["Profit_Margin_Pct"] = ((df["Profit"] / df["Revenue"]) * 100).round(2)

df[
    [
        "Transaction_ID",
        "Date",
        "Month_Name",
        "Revenue",
        "Profit",
        "Profit_Margin_Pct",
    ]
].head()

,Transaction_ID,Date,Month_Name,Revenue,Profit,Profit_Margin_Pct
0,TXN_1,2024-01-01 00:00:00,January,714.30,454.98,63.70
1,TXN_2,2024-01-01 00:30:00,January,2241.12,1448.36,64.63
2,TXN_3,2024-01-01 01:00:00,January,746.60,463.05,62.02
3,TXN_5,2024-01-01 02:00:00,January,1807.56,1515.56,83.85
4,TXN_6,2024-01-01 02:30:00,January,444.14,417.21,93.94


In [ ]:
print("=== AFTER CLEANING AUDIT ===")
print("Final Shape:", df.shape)
print("\nMissing values remaining:\n", df.isnull().sum())
print("\nFinal Data Types:")
print(df.dtypes)

# Task deliverable export karna
df.to_csv("clean_dataset.csv", index=False)
print("\nFile successfully saved as 'clean_dataset.csv'!")

=== AFTER CLEANING AUDIT ===
Final Shape: (8607, 15)

Missing values remaining:
 Transaction_ID       0
Date                 0
Customer_ID          0
Category             0
Quantity             0
Unit_Price           0
Cost_Price           0
Year                 0
Month                0
Month_Name           0
Day_Name             0
Revenue              0
Total_Cost           0
Profit               0
Profit_Margin_Pct    0
dtype: int64

Final Data Types:
Transaction_ID               object
Date                 datetime64[ns]
Customer_ID                  object
Category                     object
Quantity                      int64
Unit_Price                  float64
Cost_Price                  float64
Year                          int32
Month                         int32
Month_Name                   object
Day_Name                     object
Revenue                     float64
Total_Cost                  float64
Profit                      float64
Profit_Margin_Pct           float64
dt